# Public-source note

Paths have been sanitized. Notebook outputs are temporarily cleared in this draft while the workflow is being reorganized. For the final repository, the cleaned notebooks will be re-run and their output cells retained; any machine-specific paths printed in outputs will be removed or replaced with portable labels.


In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(ggplot2)
  library(data.table)
  library(GenomicRanges)
  library(BiocParallel)
  library(ComplexHeatmap)
  library(patchwork)
  library(reticulate)
  library(rtracklayer)
  library(tidyr)
    
library(circlize)
library(dplyr)
library(ggplot2)
library(data.table)
library(GenomicRanges)
library(BiocParallel) 
library(ComplexHeatmap)
})

In [ ]:
figure_dir_old <- "/path/to/CEBP_analysis/data/ChromBPNet_Fig4/figures_atac_chromhmm/"

# figure_dir <- "/path/to/CEBP_analysis/data/ChromBPNet_Fig4_V5/"
figure_dir <- "/path/to/CEBP_analysis/data/ChromBPNet_Fig4_V2/"
finmo_CWMs_dir <-"/path/to/CEBPB_seq/chrombpnet_hits/Combined_modisco_1000_5000000_all_CEBPB/pos_patterns/pos_patterns/merged/CWMs/"
tomtom_cwm_dir <- "/path/to/CEBPB_seq/chrombpnet_hits/Combined_modisco_1000_5000000_all_CEBPB/pos_patterns/pos_patterns/tomtom_cwm/"
modisco_html_path <- "/path/to/CEBPB_seq/chrombpnet_hits/Combined_modisco_1000_5000000_all_CEBPB/pos_patterns/pos_patterns/merged_modisco.html/motifs.html"

min_prop <- 0.05


In [ ]:
motifs_meme <- universalmotif::read_meme("/path/to/external_analysis/HumanReprogrammingMultiome/Iris/multiome_V2/tables/ChromBPNet_full//motifs.meme.txt")

In [ ]:
program_order <- c(
  "MEF",
  "MEF_d2_OSKM",
  "MEF_d2_OSKM_mCEBPA",
  "MEF_d2_OSKM_mCEBPB",
  # "MEF_d2_mCEBPA",
  # 'MEF_d2_mCEBPB' ,
  'NSC' ,
    'NSC_d2_OSKM' ,
# 'NSC_d2_mCEBPA' ,
# 'NSC_d2_mCEBPB' ,
'NSC_d2_OSKM_mCEBPA' ,
'NSC_d2_OSKM_mCEBPB' ,
  'ESC' ,
'ESC_d1_hCEBPA' ,
'ESC_d2_hCEBPA'  ,
'Bcell_Graf' ,
"Bcell_Graf_Cebpa18h",
'Bcell_Graf_OKSMd1' ,
'Bcell_Graf_OKSMd2'
)


In [ ]:
atac_state_cols2 <- paste0("A",c(
									7, # MEF open
											8, 9, # MEF CEBP open
											10, #NSC open
                                       11, # NSC CEBP open
											# 12, # NSC Cons open
											13, #ESC open
											14, #ESC CEBP open
											 16, #Bcell open
											 17 #Bcell CEBP open
											 ))
atac_state_cols2

In [ ]:
atac_state_cols <- paste0("A",c(
									# , #"Cons open"
											7, # MEF open
											8, 9, # MEF CEBP open
											10, #NSC open
                       11, # NSC CEBP open
											# 12, # NSC Cons open
											13, #ESC open
											14, #ESC CEBP open
                      # 15,  #ESC Cons open
											 16, #Bcell open
											 17 #Bcell CEBP open
                       # 18, #Bcell Cons open
											 # 19 #MEF NSC ESC CEBP open
											 ))

atac_states_split_vec <- c(
		# rep("Cons",6),
		rep("MEF",1),
		rep("MEF \nCEBP",2),
		rep("NSC",1),
		rep("NSC \nCEBP",1),
    # rep("NSC \nCons",1),
		rep("ESC",1),
		rep("ESC \nCEBP",1),
    # rep("ESC \nCons",1),
		rep("Bcell",1),
		rep("Bcell \nCEBP",1)
    # rep("Bcell \nCons",1),
		# rep("MEF \nNSC ESC\nCEBP\nopen",1)
)

atac_states_split_vec <- trimws(atac_states_split_vec)  # removes trailing spaces like "B " or "H "
atac_states_split_fac <- factor(atac_states_split_vec, levels = unique(atac_states_split_vec))  # keep first seen order


In [ ]:

atac_split_vec <- c(
		# rep("Cons",6),
		rep("MEF",4),
		rep("NSC",4),
		rep("ESC",3),
		rep("Bcell",4)
)

atac_split_vec <- trimws(atac_split_vec)  # removes trailing spaces like "B " or "H "
atac_split_fac <- factor(atac_split_vec, levels = unique(atac_split_vec))  # keep first seen order


In [ ]:
levels(atac_split_fac)

In [ ]:
unique(atac_split_vec)

In [ ]:
figure_dir

In [ ]:
atac_scaled_df <- read.table(
  paste(figure_dir_old,"/Finemo_merged_Pr_ATAC_sum_importance_over_state_size_sep_counts.csv",sep=""),
  header = TRUE,
  sep = ",",
    row.names = 1,
  stringsAsFactors = FALSE
)
head(atac_scaled_df)

In [ ]:
colnames(atac_scaled_df)

In [ ]:
nrow(atac_scaled_df)

In [ ]:
# Publication-facing motif annotation table.
# This compact TSV replaces the two historical working annotation files.
manual_ordered_joined_df <- read.delim(
  "fig5c_motif_annotations.tsv",
  header = TRUE,
  sep = "\t",
  stringsAsFactors = FALSE,
  check.names = FALSE
)

# Aliases retained so downstream cells from the original analysis remain readable.
kc_df <- manual_ordered_joined_df
manual_ordered_df <- manual_ordered_joined_df

colnames(manual_ordered_joined_df)


In [ ]:
head(kc_df)

In [ ]:
nrow(kc_df)

In [ ]:
kc_df[kc_df$Pattern == 209, ]

In [ ]:
unique(kc_df$HF)

In [ ]:
figure_dir

In [ ]:
# manual_ordered_df <- read.table(
#   paste("Finemo_PrATAC_patterns_0.05_tomtom_manual_match_manual_selected_with_KC_motifs_V4.csv",sep=""),
#   header = TRUE,
#   sep = ",",
#     # row.names = FALSE,
#   stringsAsFactors = FALSE
# )
# head(manual_ordered_df)

manual_ordered_df <- read.table(
  paste("Finemo_PrATAC_patterns_0.05_tomtom_manual_match_manual_selected.csv",sep=""),
  header = TRUE,
  sep = ",",
    # row.names = FALSE,
  stringsAsFactors = FALSE
)
head(manual_ordered_df)

In [ ]:
tail(manual_ordered_df,10)

In [ ]:
length(manual_ordered_df$cluster)

In [ ]:
length(unique(manual_ordered_df$pattern))

In [ ]:
# Keep the full publication-facing annotation table; no additional column reduction is required.


In [ ]:
head(manual_ordered_df)

In [ ]:
## keep only the rows where to_plot is 1
# manual_ordered_df <- manual_ordered_df[manual_ordered_df$to_plot == 1, ]

In [ ]:
length(manual_ordered_df$cluster)

In [ ]:
length(unique(manual_ordered_df$pattern))

In [ ]:
(unique(manual_ordered_df$pattern_num))

In [ ]:
(unique(kc_df$pattern_num))

In [ ]:
unique(kc_df$Pattern)

In [ ]:
setdiff(
  unique(manual_ordered_df$pattern_num),
  unique(kc_df$Pattern)
)

In [ ]:
length(setdiff(
  unique(manual_ordered_df$pattern_num),
  unique(kc_df$Pattern)
))

In [ ]:
setdiff(
  unique(kc_df$Pattern),
  unique(manual_ordered_df$pattern_num)
)

In [ ]:
length(setdiff(
  unique(kc_df$Pattern),
  unique(manual_ordered_df$pattern_num)
))

In [ ]:
# manual_ordered_joined_df is already loaded from fig5c_motif_annotations.tsv.


In [ ]:
head(manual_ordered_joined_df)

In [ ]:
nrow(manual_ordered_joined_df)

In [ ]:
colnames(manual_ordered_joined_df)

In [ ]:
manual_ordered_joined_df[manual_ordered_joined_df$Pattern == 140, ]

In [ ]:
manual_ordered_joined_df[manual_ordered_joined_df$Pattern == 209, ]

In [ ]:
manual_ordered_joined_df[manual_ordered_joined_df$Pattern == 140, ]$pattern

In [ ]:
manual_ordered_joined_df[manual_ordered_joined_df$pattern == 'pos_patterns.pattern_209', ]

In [ ]:
pattren_names_df <- read.table(
  paste(figure_dir_old,"/pattren_names.csv",sep=""),
  header = TRUE,
  sep = ",",
    row.names = 1,
  stringsAsFactors = FALSE
)


In [ ]:
nrow(pattren_names_df)

In [ ]:
head(pattren_names_df)

In [ ]:
manual_ordered_joined_df <- merge(manual_ordered_joined_df, pattren_names_df, by = "pattern",sort = FALSE)
rownames(manual_ordered_joined_df) <- manual_ordered_joined_df$all_matches

In [ ]:
head(manual_ordered_joined_df)

In [ ]:
nrow(atac_scaled_df[rownames(manual_ordered_joined_df),program_order])

In [ ]:
nrow(manual_ordered_joined_df)

In [ ]:
col_fun2 <- colorRamp2(
  c(-2, 0, 2),
  c("#3B4CC0", "#FFFFFF", "#E66101")
)

col_fun <- colorRamp2(
  c(-2, 0, 2),
  c("#3B4CC0", "#FFFFFF", "#B40426")
)


col_fun3 <- colorRamp2(
  c(-2, 0, 2),
  # c(-3, 0, 3),
  c("#3B4CC0", "#FFFFFF",  "red")
)

In [ ]:
head(atac_scaled_df)

In [ ]:
col_fun4 <- colorRamp2(
  c(-2, 0, 2),
  c("#2166AC", "#FFFFFF", "#B2182B")
)

col_fun5 <- colorRamp2(
  c(-2, 0, 2),
  c("#0072B2", "#F7F7F7", "#D55E00")
)
col_fun6 <- colorRamp2(
  c(-2, 0, 2),
  c("#5E3C99", "#F7F7F7", "#1B7837")
)
col_fun7 <- colorRamp2(
  c(-2, 0, 2),
  c("#2C7BB6", "#F0F0F0", "#D95F02")
)
col_fun8 <- colorRamp2(
  c(-2, 0, 2),
  c("#4A6FE3", "#FFFFFF", "#C83E4D")
)

col_fun9 <- colorRamp2(
  c(-2, 0, 2),
  c("#3B528B", "#FDE725", "#5DC863")
)

col_fun_best <- colorRamp2(
  c(-2, 0, 2),
  c("#2166AC", "#F7F7F7", "#B2182B")
)

col_fun_best2 <- colorRamp2(
  c(-2, 0, 2),
  c("#0072B2", "#F7F7F7", "#D55E00")
)

col_fun_alt1 <- colorRamp2(
  c(-2, 0, 2),
  c("#2166AC", "#F7F7F7", "#E66101")
)

col_fun_alt2 <- colorRamp2(
  c(-2, 0, 2),
  c("#2166AC", "#F7F7F7", "#FDB863")
)

col_fun_alt3 <- colorRamp2(
  c(-2, 0, 2),
  c("#2166AC", "#F7F7F7", "#7B3294")
)

col_fun_alt <- colorRamp2(
  c(-2, 0, 2),
  c("#2166AC", "#F7F7F7", "#E66101")
)

In [ ]:
atac_split_vec

In [ ]:
atac_state_cols

In [ ]:
levels(atac_states_split_fac)

In [ ]:
# outfile <- file.path( "ATAC_state_only_Motif_plot_sorted3.pdf")
outfile <- file.path( paste0("ATAC_state_only_Motif_plot_manual_sorted_",min_prop,".pdf"))

pdf(outfile, width = 12, height = 20, useDingbats=FALSE)
pdf(outfile, width = 13, height = 20, useDingbats=FALSE)
h1=ComplexHeatmap::Heatmap(atac_scaled_df[rownames(manual_ordered_joined_df),program_order],
	cluster_columns=F, cluster_rows=F,cluster_column_slices = FALSE,
  col = col_fun_best,
	row_split = manual_ordered_joined_df$cluster,border = TRUE,gap = unit(1, "mm"),
       column_split = atac_split_fac,
	heatmap_legend_param = list(title = "Scaled((sum_importance))", border = "black", title_gp = gpar(col = "black", fontsize = 10)),
	# width = unit(6, "null"))
    width = unit(4, "null"))

h2=ComplexHeatmap::Heatmap(atac_scaled_df[rownames(manual_ordered_joined_df),atac_state_cols],
	cluster_columns=F, cluster_rows=F,cluster_column_slices = FALSE,
  col = col_fun_alt,
	row_split = manual_ordered_joined_df$cluster,border = TRUE,gap = unit(1, "mm"),
	column_split = atac_states_split_fac,
    
	heatmap_legend_param = list(title = "Scaled((sum_importance_over_state_size))", border = "black", title_gp = gpar(col = "black", fontsize = 10)),
	# width = unit(5, "null"))
    width = unit(4, "null"))



ht <- h1 + h2 
# ht <- h2 + h3

draw(ht, gap = unit(3, "mm"))
dev.off()
message("Saved: ", outfile)
print(ht)

In [ ]:
levels(atac_state_cols)

In [ ]:
levels(program_order)

In [ ]:
head(manual_ordered_joined_df)

In [ ]:
ordered_patterns <- manual_ordered_joined_df$pattern
ordered_Selected_TF <- manual_ordered_joined_df$Recommended
ordered_pattern_num <- manual_ordered_joined_df$pattern_num
ordered_pattern_matched_motif <- manual_ordered_joined_df$matched_motif
ordered_pattern_matched_Orientation <- manual_ordered_joined_df$Orientation

In [ ]:

motif_list_modisco <- list()

for(pattern_sel in ordered_patterns){
	motif_modisco_fc <- read.table(paste0(finmo_CWMs_dir,pattern_sel,"/modisco_fc.txt")) 

	rownames(motif_modisco_fc) <- c("A","C","G","T")

	colnames(motif_modisco_fc) <- 1:ncol(motif_modisco_fc)
	motif_list_modisco[[paste("Modisco_FC",pattern_sel)]] <- as.matrix(motif_modisco_fc)
}


In [ ]:

fix_pwm <- function(x) {
  if (is.null(x) || length(x) == 0) return(NULL)

  # Convert data.frame to matrix
  if (is.data.frame(x)) x <- as.matrix(x)

  # If it is not matrix now, cannot use as PWM
  if (!is.matrix(x)) return(NULL)

  # If transposed (width x 4), transpose to 4 x width
  if (nrow(x) != 4 && ncol(x) == 4) x <- t(x)

  # Must be 4 rows for DNA
  if (nrow(x) != 4) return(NULL)

  # Force rownames
  rownames(x) <- c("A", "C", "G", "T")

  # Ensure numeric
  storage.mode(x) <- "double"

  x
}

fix_list <- function(lst) {
  out <- lapply(lst, fix_pwm)
  # Drop NULLs
  out <- out[!vapply(out, is.null, logical(1))]
  out
}


In [ ]:


motif_list_modisco <- fix_list(motif_list_modisco)


In [ ]:

library(ggplot2)
library(ggseqlogo)
library(patchwork)

In [ ]:

# figure_file <-   "Motifs_logs_modisco_manual_order.pdf"

# # helper once, outside the loop
# logo_panel <- function(m, title) {
#   if (is.null(m)) {
#     ggplot() + theme_void() + ggtitle(paste(title, "(missing)"))
#   } else {
#     ggplot() +
#       ggseqlogo::geom_logo(list(m), method = "custom") +
#       ggseqlogo::theme_logo() +
#       ggtitle(title) +
#       theme(
#         plot.title = element_text(size = 8, hjust = 0.5),
#         axis.title.x = element_blank(),
#         axis.title.y = element_blank(),
#         axis.text.x  = element_blank(),
#         axis.text.y  = element_blank(),
#         axis.ticks.x = element_blank(),
#         axis.ticks.y = element_blank(),
#         panel.grid   = element_blank(),
#         panel.border = element_blank(),
#         plot.margin  = margin(0, 0, 0, 0)
#       )
#   }
# }

# get_tf_from_key <- function(k) {
#   if (is.na(k) || length(k) == 0) return("missing")
#   parts <- strsplit(k, " ")[[1]]
#   if (length(parts) >= 3) parts[3] else "missing"
# }

# rows <- vector("list", length(ordered_patterns))

# for (i in seq_along(ordered_patterns)) {
#   pattern_sel <- ordered_patterns[i]

#   key_fc     <- paste("Modisco_FC", pattern_sel)

#   fc     <- motif_list_modisco[[key_fc]]



#   p2 <- logo_panel(fc,    paste(ordered_Selected_TF[i],ordered_pattern_num[i]))

#   rows[[i]] <- ( p2 ) +
#     plot_annotation(title = pattern_sel) &
#     theme(
#       plot.title = element_text(size = 9, hjust = 0),
#       plot.margin = margin(0, 0, 0, 0)
#     )
# }

# # stack all rows vertically into one figure
# fig_all <- wrap_plots(rows, ncol = 1)

# # set page height based on number of motifs
# n <- length(rows)
# pdf(figure_file, width = 5, height = 80, useDingbats = FALSE)
# print(fig_all)
# dev.off()



In [ ]:
# pdf(outfile, width = 20, height = 30, useDingbats=FALSE)

## Plot Motif Count 

In [ ]:
finemo_hits <- fread(
  file.path( "/path/to/CEBPB_seq/chrombpnet_hits/Combined_modisco_1000_5000000_all_CEBPB/pos_patterns/pos_patterns/merged/hits.tsv")
)

In [ ]:
head(finemo_hits)

In [ ]:
length(unique(finemo_hits$motif_name))


In [ ]:
nrow(finemo_hits)

In [ ]:

# Clean program label (Pr) from peak_name
# Example peak_name: "MEF_mm10_chr1_..." or "MEF_ATAC_mm10_..."
finemo_hits$Pr <- finemo_hits$peak_name
finemo_hits$Pr <- sub("(_mm10.*)$", "", finemo_hits$Pr)
finemo_hits$Pr <- sub("[-_]?ATAC.*$", "", finemo_hits$Pr, ignore.case = TRUE)
finemo_hits$Pr <- sub("[-_]?merged*$", "", finemo_hits$Pr, ignore.case = TRUE)
finemo_hits$Pr <- sub("[-_]?flag$", "", finemo_hits$Pr, ignore.case = TRUE)

In [ ]:
finemo_hits <- finemo_hits %>%
  filter(Pr %in% program_order)

In [ ]:
unique(finemo_hits$Pr)
length(unique(finemo_hits$Pr))

In [ ]:
head(manual_ordered_joined_df)

In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)
library(scales)

row_df <- manual_ordered_joined_df %>%
  select(pattern, cluster) %>%
  mutate(pattern = as.character(pattern))

ordered_patterns <- row_df$pattern

In [ ]:
library(ComplexHeatmap)
library(circlize)
library(scales)

#Prepare matrix
ordered_patterns <- manual_ordered_joined_df$pattern

finemo_hits2 <- finemo_hits %>%
  filter(motif_name %in% ordered_patterns)

motif_counts <- finemo_hits2 %>%
  count(motif_name, name = "n")

plot_counts_df <- manual_ordered_joined_df %>%
  select(pattern, cluster) %>%
  left_join(motif_counts, by = c("pattern" = "motif_name")) %>%
  mutate(n = replace_na(n, 0))

In [ ]:
#Create matrix
count_mat <- matrix(
  plot_counts_df$n,
  ncol = 1,
  dimnames = list(plot_counts_df$pattern, "# instances")
)

In [ ]:
head(count_mat)

In [ ]:
count_mat_log <- log10(count_mat + 1)

In [ ]:
col_fun_counts <- circlize::colorRamp2(
  c(
    min(count_mat_log),
    median(count_mat_log),
    max(count_mat_log)
  ),
  c("#F7FBFF", "#6BAED6", "#08306B")
)

In [ ]:
threshold <- 1000000

Motifs_counts_ht <- Heatmap(
  count_mat_log,
  name = "Total instances\nacross cell types",
  col = col_fun_counts,

  cluster_rows = FALSE,
  cluster_columns = FALSE,
    show_row_names = FALSE,

  row_split = plot_counts_df$cluster,
  gap = unit(1, "mm"),
    row_title = NULL,
    row_title_gp = gpar(fontsize = 0),

  rect_gp = gpar(col = "black", lwd = 0.5),
  border = TRUE,
    show_heatmap_legend = FALSE,

  cell_fun = function(j, i, x, y, width, height, fill) {

    val <- count_mat[i, j]

    if(!is.na(val)){

      grid.text(
        scales::comma(val),
        x,
        y,
        gp = gpar(
          fontsize = 8,
          col = ifelse(val > threshold, "white", "black")
        )
      )
    }

  },

 width = unit(1.5, "null"),

  heatmap_legend_param = list(
    title = "Total instances\nacross cell types",
    title_position = "topcenter"
  )
)

In [ ]:
figure_file <- "Motifs_counts_manual_order.pdf"

pdf(figure_file, width = 2.5, height = 50, useDingbats = FALSE)

draw(Motifs_counts_ht)

dev.off()

In [ ]:
print(Motifs_counts_ht)

In [ ]:
outfile <- file.path(paste0("ATAC_state_only_Motif_plot_manual_sorted_",min_prop,"_with_counts.pdf"))

pdf(outfile, width = 14, height = 30, useDingbats = FALSE)

ht <- Motifs_counts_ht + h1 + h2

draw(
  ht,
  gap = unit(3, "mm"),
  heatmap_legend_side = "bottom"
)

dev.off()

message("Saved: ", outfile)

In [ ]:
print(ht)

In [ ]:
head(finemo_hits)

## Plot count for each sample 

In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(tibble)
  library(ComplexHeatmap)
  library(circlize)
  library(scales)
  library(grid)
})

# Keep motif order from your manual table
ordered_patterns <- manual_ordered_joined_df$pattern

# Optional: if you already have a desired Pr order, use it
# Example:
# program_order <- c("A1","A2","A3","A4","A5","A6","A7","A8","A9","A10","A11","A12")

# Count motif_name x Pr
motif_pr_counts <- finemo_hits %>%
  filter(motif_name %in% ordered_patterns) %>%
  count(motif_name, Pr, name = "n")

# Make complete motif x Pr table
plot_counts_df <- motif_pr_counts %>%
  pivot_wider(
    names_from = Pr,
    values_from = n,
    values_fill = 0
  )

# Add missing motifs back in the desired row order
plot_counts_df <- tibble(motif_name = ordered_patterns) %>%
  left_join(plot_counts_df, by = "motif_name")

plot_counts_df <- plot_counts_df %>%
  select(motif_name, all_of(program_order))

In [ ]:
head(plot_counts_df)

In [ ]:
library(dplyr)

plot_counts_pct <- plot_counts_df %>%
  mutate(
    total = rowSums(across(where(is.numeric)))
  ) %>%
  mutate(
    across(where(is.numeric), ~ .x / total * 100)
  ) %>%
  select(-total)

In [ ]:
head(plot_counts_pct)


In [ ]:
plot_counts_pct %>%
  filter(motif_name == "pos_patterns.pattern_160")

In [ ]:
plot_counts_df %>%
  filter(motif_name == "pos_patterns.pattern_160")

In [ ]:
plot_counts_df %>%
  filter(motif_name == "pos_patterns.pattern_160") %>%
  select(-motif_name) %>%
  rowSums()

In [ ]:
# zlim <- max(abs(z_mat), na.rm = TRUE)
# col_fun_pct_z <- circlize::colorRamp2(
#   c(-zlim, 0, zlim),
#   c("navy", "grey95", "firebrick3")
# )

In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tibble)
  library(ComplexHeatmap)
  library(circlize)
  library(scales)
  library(grid)
})

# Build percentage matrix
pct_mat <- plot_counts_pct %>%
  column_to_rownames("motif_name") %>%
  as.matrix()

# Make sure rows follow the desired motif order
pct_mat <- pct_mat[ordered_patterns, , drop = FALSE]

# Optional: make sure columns follow program_order
pct_mat <- pct_mat[, program_order, drop = FALSE]

# Row wise z score for coloring
z_mat <- t(scale(t(pct_mat)))

# Replace NA rows (happens when a row has zero variance)
z_mat[is.na(z_mat)] <- 0

# Color scale for z scores
zlim <- max(abs(z_mat), na.rm = TRUE)

col_fun_pct_z <- circlize::colorRamp2(
  c(-zlim, 0, zlim),
  c("#3B4CC0", "#FFFFFF", "#B40426")
)

# zlim <- max(abs(z_mat), na.rm = TRUE)

# col_fun_pct_z <- circlize::colorRamp2(
#   c(-zlim, 0, zlim),
#   c("navy", "grey95", "firebrick3")
# )

col_fun_pct_z <- circlize::colorRamp2(
  c(0, zlim),
  c("grey95", "firebrick3")
)


Pct_ht <- Heatmap(
  z_mat,
  name = "Row z score",
  col = col_fun_pct_z,

  cluster_rows = FALSE,
  cluster_columns = FALSE,
  show_row_names = TRUE,

  row_split = manual_ordered_joined_df$cluster,
  gap = unit(1, "mm"),
  row_title = NULL,
  row_title_gp = gpar(fontsize = 0),

  rect_gp = gpar(col = "black", lwd = 0.5),
  border = TRUE,

  cell_fun = function(j, i, x, y, width, height, fill) {
    val <- pct_mat[i, j]

    if (!is.na(val)) {
      grid.text(
        label = paste0(round(val)),
        x = x,
        y = y,
        gp = gpar(fontsize = 7, col = "black")
      )
    }
  },

  heatmap_legend_param = list(
    title = "Row z score",
    title_position = "topcenter"
  )
)



In [ ]:
figure_file <- "Motifs_Pct_per_sample_manual_order.pdf"

pdf(figure_file, width = 9, height = 30, useDingbats = FALSE)

draw(Pct_ht)

dev.off()

In [ ]:
# suppressPackageStartupMessages({
#   library(dplyr)
#   library(tibble)
#   library(ComplexHeatmap)
#   library(circlize)
#   library(grid)
# })

# # Percentage matrix for labels
# pct_mat <- plot_counts_pct %>%
#   column_to_rownames("motif_name") %>%
#   as.matrix()

# pct_mat <- pct_mat[ordered_patterns, program_order, drop = FALSE]

# # Row wise z score for coloring
# z_mat <- t(scale(t(pct_mat)))
# z_mat[is.na(z_mat)] <- 0

# # Symmetric color scale around 0
# zlim <- max(abs(z_mat), na.rm = TRUE)

# col_fun_pct_z <- circlize::colorRamp2(
#   c(-zlim, 0, zlim),
#   c("#3B4CC0", "#FFFFFF", "#B40426")
# )

# Pct_ht <- Heatmap(
#   z_mat,
#   name = "Row z score",
#   col = col_fun_pct_z,

#   cluster_rows = FALSE,
#   cluster_columns = FALSE,
#   show_row_names = FALSE,

#   row_split = manual_ordered_df$cluster,
#   gap = unit(1, "mm"),
#   row_title = NULL,
#   row_title_gp = gpar(fontsize = 0),

#   rect_gp = gpar(col = "black", lwd = 0.5),
#   border = TRUE,
#   show_heatmap_legend = TRUE,

#   cell_fun = function(j, i, x, y, width, height, fill) {
#     val <- pct_mat[i, j]

#     if (!is.na(val)) {
#       grid.text(
#         label = paste0(as.integer(val), "%"),
#         x = x,
#         y = y,
#         gp = gpar(fontsize = 7, col = "black")
#       )
#     }
#   },

#   heatmap_legend_param = list(
#     title = "Row z score",
#     title_position = "topcenter",
#     legend_height = unit(4, "cm")
#   )
# )

# Pct_ht

In [ ]:
# figure_file <- "Motifs_Pct_per_sample_manual_order.pdf"

# pdf(figure_file, width = 6, height = 50, useDingbats = FALSE)

# draw(Pct_ht)

# dev.off()

In [ ]:
# Replace any remaining NA with 0
plot_counts_df[is.na(plot_counts_df)] <- 0

# Keep only columns that are actual Pr columns
pr_cols <- setdiff(colnames(plot_counts_df), "motif_name")

# If you want a specific Pr order, use this
# pr_cols <- intersect(program_order, pr_cols)

# Build count matrix
count_mat <- plot_counts_df %>%
  select(motif_name, all_of(pr_cols)) %>%
  column_to_rownames("motif_name") %>%
  as.matrix()

# Make sure row order matches manual_ordered_joined_df
count_mat <- count_mat[ordered_patterns, , drop = FALSE]

count_mat <- count_mat[, program_order, drop = FALSE]

# Log transform for coloring only
count_mat_log <- log10(count_mat + 1)


In [ ]:
head(count_mat_log)

In [ ]:



# # Color function
# vals <- as.numeric(count_mat_log)
# vmin <- min(vals, na.rm = TRUE)
# vmed <- median(vals, na.rm = TRUE)
# vmax <- max(vals, na.rm = TRUE)

# if (vmin == vmax) {
#   col_fun_counts <- circlize::colorRamp2(
#     c(vmin, vmax + 1e-6),
#     c("#F7FBFF", "#08306B")
#   )
# } else {
#   col_fun_counts <- circlize::colorRamp2(
#     c(vmin, vmed, vmax),
#     c("#F7FBFF", "#6BAED6", "#08306B")
#   )
# }

# threshold <- 1000000

# Motifs_counts_ht2 <- Heatmap(
#   count_mat_log,
#   name = "Instances",
#   col = col_fun_counts,

#   cluster_rows = FALSE,
#   cluster_columns = FALSE,
#   show_row_names = TRUE,

#   row_split = manual_ordered_df$cluster,
#   gap = unit(1, "mm"),
#   row_title = NULL,
#   row_title_gp = gpar(fontsize = 0),

#   rect_gp = gpar(col = "black", lwd = 0.5),
#   border = TRUE,
#   show_heatmap_legend = FALSE,

#   cell_fun = function(j, i, x, y, width, height, fill) {
#     val <- count_mat[i, j]

#     if (!is.na(val) && val > 0) {
#       grid.text(
#         scales::comma(val),
#         x, y,
#         gp = gpar(
#           fontsize = 7,
#           col = ifelse(val > threshold, "white", "black")
#         )
#       )
#     }
#   },

#   heatmap_legend_param = list(
#     title = "Total instances",
#     title_position = "topcenter"
#   )
# )

# # Motifs_counts_ht

In [ ]:
# figure_file <- "Motifs_counts_per_sample_manual_order.pdf"

# pdf(figure_file, width = 10, height = 50, useDingbats = FALSE)

# draw(Motifs_counts_ht2)

# dev.off()

In [ ]:
length(colnames(count_mat_log))*2.5

In [ ]:
# motif_pr_counts <- finemo_hits %>%
#   filter(motif_name %in% ordered_patterns) %>%
#   count(motif_name, Pr, name = "n")

In [ ]:
# head(motif_pr_counts)

In [ ]:
# library(tidyr)

# motif_pr_mat <- motif_pr_counts %>%
#   pivot_wider(
#     names_from = Pr,
#     values_from = n,
#     values_fill = 0
#   )

In [ ]:
# head(motif_pr_mat)

In [ ]:
# motif_pr_mat <- motif_pr_mat %>%
#   right_join(
#     tibble(motif_name = ordered_patterns),
#     by = "motif_name"
#   ) %>%
#   replace_na(list_fill(0))

In [ ]:
# library(dplyr)
# library(tidyr)
# library(ComplexHeatmap)
# library(circlize)
# library(grid)
# library(scales)


# # ordered motif list
# ordered_patterns <- manual_ordered_df$pattern

# # keep only motifs you want in the final plot
# finemo_hits2 <- finemo_hits %>%
#   filter(motif_name %in% ordered_patterns)

# # count motif instances per motif per program
# motif_pr_counts <- finemo_hits2 %>%
#   count(motif_name, Pr, name = "n") %>%
#   complete(
#     motif_name = ordered_patterns,
#     Pr = program_order,
#     fill = list(n = 0)
#   )

# # convert to wide matrix
# motif_pr_wide <- motif_pr_counts %>%
#   tidyr::pivot_wider(
#     names_from = Pr,
#     values_from = n
#   )

# # keep row order same as manual_ordered_df
# motif_pr_wide <- manual_ordered_df %>%
#   select(pattern, cluster) %>%
#   left_join(motif_pr_wide, by = c("pattern" = "motif_name"))

# # matrix of raw counts
# count_mat_pr <- as.matrix(motif_pr_wide[, program_order])
# rownames(count_mat_pr) <- motif_pr_wide$pattern

# # log transform for color only
# count_mat_pr_log <- log10(count_mat_pr + 1)

# # color scale
# col_fun_counts_pr <- circlize::colorRamp2(
#   c(
#     min(count_mat_pr_log, na.rm = TRUE),
#     median(count_mat_pr_log, na.rm = TRUE),
#     max(count_mat_pr_log, na.rm = TRUE)
#   ),
#   c("#F7FBFF", "#6BAED6", "#08306B")
# )

# threshold_pr <- 100000

# Motifs_counts_by_Pr_ht <- Heatmap(
#   count_mat_pr_log,
#   name = "Instances per program",
#   col = col_fun_counts_pr,

#   cluster_rows = FALSE,
#   cluster_columns = FALSE,

#   row_split = motif_pr_wide$cluster,
#   gap = unit(1, "mm"),
#   row_title = NULL,
#   row_title_gp = gpar(fontsize = 0),

#   show_row_names = FALSE,
#   border = TRUE,
#   rect_gp = gpar(col = "black", lwd = 0.5),

#   cell_fun = function(j, i, x, y, width, height, fill) {
#     val <- count_mat_pr[i, j]

#     if (!is.na(val) && val > 0) {
#       grid.text(
#         scales::comma(val),
#         x, y,
#         gp = gpar(
#           fontsize = 7,
#           col = ifelse(val > threshold_pr, "white", "black")
#         )
#       )
#     }
#   },

#   heatmap_legend_param = list(
#     title = "Count per program",
#     title_position = "topcenter"
#   )
# )

# Motifs_counts_by_Pr_ht

In [ ]:
# head(manual_ordered_df)

In [ ]:
# manual_ordered_df$Selected_TF

In [ ]:
head(manual_ordered_joined_df)

In [ ]:
library(ComplexHeatmap)
library(grid)

n_rows <- nrow(manual_ordered_joined_df)
row_split_vec <- manual_ordered_joined_df$cluster

make_text_heatmap <- function(labels, title, width_cm = 3, fontsize = 8) {
  
  mat <- matrix(
    1,
    nrow = length(labels),
    ncol = 1,
    dimnames = list(rownames(manual_ordered_joined_df), title)
  )
  
  Heatmap(
    mat,
    name = title,
    col = c("1" = "white"),
    
    cluster_rows = FALSE,
    cluster_columns = FALSE,
    
    row_split = row_split_vec,
    gap = unit(1, "mm"),
    
    show_row_names = FALSE,
    show_column_names = FALSE,
    row_title = NULL,
    show_heatmap_legend = FALSE,
    
    rect_gp = gpar(col = "black", lwd = 0.5),
    border = TRUE,
    
    cell_fun = function(j, i, x, y, width, height, fill) {
      grid.text(
        labels[i],
        x = x,
        y = y,
        just = "center",
        gp = gpar(fontsize = fontsize, col = "black")
      )
    },
    
    width = unit(width_cm, "cm")
  )
}

Selected_TF_ht <- make_text_heatmap(
  labels = as.character(manual_ordered_joined_df$Motif),
  title = "Selected_TF",
  width_cm = 3
)

pattern_num_ht <- make_text_heatmap(
  labels = as.character(manual_ordered_joined_df$Pattern),
  title = "pattern_num",
  width_cm = 1
)

outfile <- "TF_pattern_table_bordered.pdf"

pdf(
  outfile,
  width = 5.5,
  height = n_rows * 0.35,
  useDingbats = FALSE
)

ht <- Selected_TF_ht + pattern_num_ht
draw(ht, gap = unit(2, "mm"))

dev.off()
message("Saved: ", outfile)

In [ ]:
print(ht)

In [ ]:
colnames(manual_ordered_joined_df)

length(manual_ordered_joined_df$Selected_TF)
nrow(manual_ordered_joined_df)

In [ ]:
nrow(manual_ordered_joined_df)

length(manual_ordered_joined_df$Recommended)
length(manual_ordered_joined_df$Pattern)
length(manual_ordered_joined_df$cluster)

In [ ]:
# outfile <- file.path(paste0("ATAC_state_only_Motif_plot_manual_sorted_",min_prop,"_with_counts_and_table.pdf"))

# pdf(outfile, width = 16, height = 30, useDingbats = FALSE)

# ht <- Selected_TF_ht + pattern_num_ht + Motifs_counts_ht + h1 + h2

# draw(
#   ht,
#   gap = unit(3, "mm"),
#   heatmap_legend_side = "bottom"
# )

# dev.off()

# message("Saved: ", outfile)

In [ ]:
outfile <- file.path(paste0("ATAC_state_only_Motif_plot_manual_sorted_",min_prop,"_with_counts_and_table.pdf"))

# pdf(outfile, width = 12, height = 6, useDingbats = FALSE)

# pdf(outfile, width = 14, height = 30, useDingbats = FALSE)

# pdf(outfile, width = 14, height = 24, useDingbats = FALSE)
pdf(outfile, width = 15, height = 24, useDingbats = FALSE)

ht <- Selected_TF_ht + pattern_num_ht + Motifs_counts_ht + h1 + h2

draw(
  ht,
  gap = unit(3, "mm"),
  heatmap_legend_side = "bottom"
)

dev.off()

message("Saved: ", outfile)

In [ ]:
print(ht)

In [ ]:
print(pattern_num_ht)

In [ ]:
print(Selected_TF_ht)

In [ ]:
old_height <- 30
old_nrows <- 100
new_nrows <- 50

new_height <- old_height * new_nrows / old_nrows

new_height